# 04 - Rendering the interactive composition viewer

This notebook turns the grid from notebook 03 into a single self-contained HTML page: an
equirectangular lunar albedo basemap with one hoverable marker per grid cell, each carrying
all eight element/silicon ratios and their uncertainties.

The only decision that really matters here is the renderer. Plotly's default `Scatter`
trace emits one DOM node per point; at ten thousand points the page is already sluggish and
at the scale of this dataset - just over 900,000 cells - it never finishes laying out.
`Scattergl` uploads the same points to the GPU as vertex buffers instead, and draws them in
parallel, so pan, zoom and hover all stay interactive. The hover payload rides along in the
`customdata` buffer, which is why the tooltip can show eight ratios without a single
callback or round trip.

Markers are drawn at one pixel. That is deliberate: the visible density of the map then
reads as genuine coverage rather than as a smoothing artefact, with mare regions appearing
solid where many passes overlap and polar tracks appearing as individual lines.

**Expected input:** the grid CSV from notebook 03, plus an equirectangular lunar basemap
image spanning -180..180 deg longitude and -90..90 deg latitude.

**Expected output:** a standalone `.html` file that opens in any browser with WebGL.

## Inputs

In [ ]:
# Dependencies: see requirements.txt.
# On a hosted runtime:  !pip install plotly pandas numpy pillow

import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from PIL import Image

# Equirectangular lunar albedo basemap. Any global mosaic works as long as it spans the
# full -180..180 / -90..90 range; see data/README.md for the source used here.
BASEMAP_PATH = os.environ.get("LUNAR_BASEMAP", "../data/basemap/lunar_albedo_equirect.png")

# Grid table written by notebook 03.
GRID_CSV = os.environ.get("GRID_CSV", "../data/interim/grid_ratios.csv")

# Destination for the rendered page.
VIEWER_HTML = os.environ.get("VIEWER_HTML", "../results/composition_viewer.html")

# Some global mosaics exceed Pillow's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

In [ ]:
# When running on a hosted notebook backed by cloud storage, mount it first.
# On a local machine this block is simply skipped.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

## Basemap geometry

Pixel coordinates come straight from a linear equirectangular mapping, so the image
dimensions are read from the file rather than hard-coded. Nothing else in this notebook
needs to know the projection.

In [ ]:
# Read the basemap and take its dimensions from the file itself.
lunar_map = Image.open(BASEMAP_PATH)
img_width, img_height = lunar_map.size

print(f"Lunar map dimensions: Width = {img_width}, Height = {img_height}")

## Loading the grid and projecting it to pixels

The assertions are worth keeping: a grid written by an older version of notebook 03, or one
truncated mid-write, fails here rather than producing a map with silently missing ratios.

Latitude is mapped with the image's y-axis convention in mind - image rows increase
downwards while latitude increases upwards.

In [ ]:
df = pd.read_csv(GRID_CSV)

# Notebook 03 writes lowercase coordinate column names; accept either spelling
# so the two stages connect without an intermediate rename.
df = df.rename(columns={"latitude": "Latitude", "longitude": "Longitude"})
print(f"Loaded {len(df)} points in the full dataset.")

# Uncomment to render a small subsample while iterating on the layout.
#df = df.sample(n=1000, random_state=42)

# Fail loudly if the grid is missing anything the tooltip needs.
assert 'Latitude' in df.columns, "Latitude column not found!"
assert 'Longitude' in df.columns, "Longitude column not found!"
required_ratios = [
    "O/Si_ratio", "Na/Si_ratio", "Mg/Si_ratio", "Al/Si_ratio",
    "Ca/Si_ratio", "Ti/Si_ratio", "Mn/Si_ratio", "Fe/Si_ratio"
]
assert all(ratio in df.columns for ratio in required_ratios), "One or more ratio columns are missing!"

# Equirectangular projection to pixel space. The y expression is matched to the
# axis range set further down, where y runs 0..img_height with the image anchored
# at the top; changing one without the other flips the map vertically.
df['x_pixel'] = ((df['Longitude'] + 180) / 360) * img_width
df['y_pixel'] = ((df['Latitude']-90) / 180) * img_height

print("Pixel coordinates calculated for latitude and longitude.")

## Figure size

The figure is rendered at a fraction of the basemap's native resolution; the aspect ratio
is derived rather than assumed so that a different mosaic does not distort the map.

In [ ]:
# Fraction of the basemap's native pixel size used for the on-screen figure.
scaling_factor = 0.4
fig_width = img_width * scaling_factor
fig_height = fig_width / (img_width / img_height)

## Building and exporting the map

`customdata` carries the coordinates and all eight ratio strings for every point. It is
uploaded once alongside the vertex buffer, which is what lets the tooltip resolve instantly
at any zoom level.

In [ ]:
# Build the figure.
fig = go.Figure()

# Basemap sits underneath every trace.
fig.add_layout_image(
    dict(
        source=lunar_map,
        x=0,
        y=img_height,
        xref="x",
        yref="y",
        sizex=img_width,
        sizey=img_height,
        xanchor="left",
        yanchor="top",
        layer="below",
    )
)

# One GPU-rendered point per grid cell, with its full readout attached.
fig.add_trace(
    go.Scattergl(
        x=df['x_pixel'],
        y=df['y_pixel'],
        mode="markers",
        marker=dict(size=1, color='blue', opacity=0.25),
        hovertemplate=(
            "Lat: %{customdata[0]:.2f}<br>"
            "Lon: %{customdata[1]:.2f}<br>"
            "O/Si_ratio: %{customdata[2]}<br>"
            "Na/Si_ratio: %{customdata[3]}<br>"
            "Mg/Si_ratio: %{customdata[4]}<br>"
            "Al/Si_ratio: %{customdata[5]}<br>"
            "Ca/Si_ratio: %{customdata[6]}<br>"
            "Ti/Si_ratio: %{customdata[7]}<br>"
            "Mn/Si_ratio: %{customdata[8]}<br>"
            "Fe/Si_ratio: %{customdata[9]}<extra></extra>"
        ),
        customdata=df[[
            "Latitude", "Longitude", "O/Si_ratio", "Na/Si_ratio", "Mg/Si_ratio",
            "Al/Si_ratio", "Ca/Si_ratio", "Ti/Si_ratio", "Mn/Si_ratio", "Fe/Si_ratio"
        ]].values
    )
)

aspect_ratio = img_width / img_height

# Figure dimensions, in pixels.
fig_width = img_width * 0.4
fig_height = fig_width / aspect_ratio

# Axes are pixel coordinates on the basemap and are hidden from the viewer.
fig.update_layout(
    width=fig_width,
    height=fig_height,
    xaxis=dict(range=[0, img_width], visible=False),
    yaxis=dict(range=[0, img_height], visible=False),
    title="Lunar surface composition - element/Si ratios",
)

# Without this the markers drift off the basemap as the window is resized.
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1
)

# Write a self-contained page. The result is large - the whole dataset is inlined -
# but it needs no server and no network access to open.
os.makedirs(os.path.dirname(VIEWER_HTML), exist_ok=True)
fig.write_html(VIEWER_HTML)

print(f"Viewer written to {VIEWER_HTML}")